In [ ]:
import os
import json
import time
import requests
import urllib3
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
BASE_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(BASE_DIR, "dados_netpa")
DOWNLOAD_DIR = os.path.join(OUTPUT_DIR, "downloads_2.0")

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

options = Options()
prefs = {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True,
    "profile.default_content_setting_values.automatic_downloads": 1,
    "profile.content_settings.exceptions.automatic_downloads.*.setting": 1
}
options.add_experimental_option("prefs", prefs)

service = Service(r"edgedriver_win64\msedgedriver.exe") # Path to your Edge WebDriver
driver = webdriver.Edge(service=service, options=options)

DOWNLOAD_MAP = {
    "calendario_candidaturas": {
        "tipo": "expandir",
        "xpath": "//div[contains(@class, 'x-grid-group-title')]"
    },
    "matricula_inscricao": {
        "tipo": "print_multiplo",
        "xpath": "//a[contains(@class, 'print')]" 
    },    
    "situacao_curricular": {
        "tipo": "excel_multiplo", 
        "xpath": "//button[@data-qtip='Exportar para Excel']"
    },
    "inscricao_exames": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "consultar_candidaturas": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "bolsa": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "faltas": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "formacao_avancada": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "estagios": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "pedidos_revisao_notas": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "requisicoes_documentos": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "atividades_curriculares": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "atividades_extracurriculares": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "competencias_linguisticas": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "documentos_faturacao": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "pagamentos": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "associacao_entidades_pagadoras": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "planos_pagamento": {"tipo": "excel", "xpath": "//button[@data-qtip='Exportar para Excel']"},
    "notas": {"tipo": "download_direto", "xpath": "//span[text()='Imprimir']/ancestor::button"},
}


def baixar_pdf_da_janela(driver, nome_base, contador):
    try:
        url_pdf = driver.current_url
        
        session = requests.Session()
        for cookie in driver.get_cookies():
            session.cookies.set(cookie['name'], cookie['value'])
        response = session.get(url_pdf, stream=True, verify=False)
        
        if response.status_code == 200:
            nome_fich = f"{nome_base}_{contador}.pdf"
            
            with open(os.path.join(DOWNLOAD_DIR, nome_fich), 'wb') as f:
                
                for chunk in response.iter_content(chunk_size=8192): 
                    f.write(chunk)
    except: pass

def fechar_popup_later(driver):
    try:
        WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.ID, "notificacoesNetpa_OK-btnEl"))).click()
        time.sleep(0.5)
    except: pass

def obter_texto_total(driver):

    texto_acumulado = ""
    
    # ExTJS Grids
    try:
        celulas = driver.find_elements(By.CLASS_NAME, "x-grid-cell-inner")
        if celulas:
            texto_acumulado += "\n--- GRELHA EXTJS ---\n"
            linha_temp = ""
            for i, c in enumerate(celulas):
                t = c.text.strip()
                if t: linha_temp += t + " | "
                if (i+1) % 8 == 0: 
                    texto_acumulado += linha_temp + "\n"
                    linha_temp = ""
            texto_acumulado += linha_temp + "\n"
    except: pass

    # HTML Tables
    try:
        tabelas = driver.find_elements(By.TAG_NAME, "table")
        for idx, tbl in enumerate(tabelas):
            if tbl.size['height'] > 20:
                texto_acumulado += f"\n--- TABELA {idx+1} ---\n"
                linhas = tbl.find_elements(By.TAG_NAME, "tr")
                for tr in linhas:
                    texto_acumulado += tr.text.strip().replace("\n", " | ") + "\n"
    except: pass

    # Lines outside tables
    try:
        
        if len(texto_acumulado) < 200:
            todas_linhas = driver.find_elements(By.TAG_NAME, "tr")
            if len(todas_linhas) > 0:
                texto_acumulado += "\n--- LINHAS SOLTAS ---\n"
                for tr in todas_linhas:
                    txt = tr.text.strip()
                    if len(txt) > 5: 
                        texto_acumulado += txt.replace("\n", " | ") + "\n"
    except: pass

    # Document Body
    try:
        texto_body = driver.execute_script("return document.body.innerText;").strip()
        if len(texto_body) > len(texto_acumulado) + 100:
            texto_acumulado = "--- TEXTO GERAL ---\n" + texto_body + "\n\n" + texto_acumulado
    except: pass

    # Iframes
    try:
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        for i, iframe in enumerate(iframes):
            driver.switch_to.frame(iframe)
            txt_if = obter_texto_total(driver)
            if len(txt_if) > 50:
                texto_acumulado += f"\n--- IFRAME {i+1} ---\n{txt_if}\n"
            driver.switch_to.default_content()
    except:
        driver.switch_to.default_content()

    return texto_acumulado

def tratar_interacoes_e_downloads(driver, page_name, config):

    janela_principal = driver.current_window_handle
    texto_expansao = ""
    
    try:
        WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.XPATH, config['xpath'])))
        elems = driver.find_elements(By.XPATH, config['xpath'])
        total = len(elems)
        
        if "multiplo" not in config['tipo'] and "expandir" not in config['tipo']:
            total = 1

        for i in range(total):
            try:
                elems = driver.find_elements(By.XPATH, config['xpath'])
                if i >= len(elems): break
                el = elems[i]
                
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
                time.sleep(1)

                if config['tipo'] == "expandir":
                    try: nome = el.text.replace("\n", " ")[:20]
                    except: nome = f"Item {i}"

                    driver.execute_script("arguments[0].click();", el)
                    time.sleep(5)
                    txt = obter_texto_total(driver)
                    if len(txt) > 50: 
                        texto_expansao += f"\n\n=== {nome} ===\n{txt}"
                
                elif config['tipo'] in ["excel", "download_direto", "excel_multiplo"]:
                    driver.execute_script("arguments[0].click();", el)
                    time.sleep(4)
                
                elif "print" in config['tipo']:
                    driver.execute_script("arguments[0].click();", el)
                    time.sleep(6)
                    if len(driver.window_handles) > 1:
                        janela_popup = [w for w in driver.window_handles if w != janela_principal][-1]
                        driver.switch_to.window(janela_popup)
                        try:
                            if "edge://" not in driver.current_url:
                                baixar_pdf_da_janela(driver, page_name, i+1)
                        except: pass
                        finally:
                            driver.close()
                            driver.switch_to.window(janela_principal)

            except Exception as e:

                if driver.current_window_handle != janela_principal:
                    if len(driver.window_handles) > 1: driver.close()
                    driver.switch_to.window(janela_principal)
    except: pass
    
    return texto_expansao

def click_and_dump(driver, xpath, filename, wait_time=15, back_times=1):
    try:
        # Navegate
        link = WebDriverWait(driver, wait_time).until(EC.presence_of_element_located((By.XPATH, xpath)))
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", link)
        driver.execute_script("arguments[0].click();", link)
        time.sleep(4)
        fechar_popup_later(driver)
        texto_final = ""

        # Situação Curricular
        if filename == "situacao_curricular":
            try:
                media_btn = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, "calcularNotaAtualfunc")))
                driver.execute_script("arguments[0].click();", media_btn)
                time.sleep(3)
                popup = WebDriverWait(driver, 5).until(EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'x-message-box-info') and contains(., 'Média Atual')]")))
                texto_final += f"\n=== MÉDIA ===\n{popup.text}\n"
            except: pass
            
            texto_final += obter_texto_total(driver)

            if filename in DOWNLOAD_MAP:
                tratar_interacoes_e_downloads(driver, filename, DOWNLOAD_MAP[filename])

        # Calendário de Candidaturas
        elif filename in DOWNLOAD_MAP and DOWNLOAD_MAP[filename]['tipo'] == "expandir":
            texto_final += tratar_interacoes_e_downloads(driver, filename, DOWNLOAD_MAP[filename])

        # Resumo Financeiro
        elif filename == "resumo_financeiro":
            texto_final += obter_texto_total(driver)
            try:
                link_det = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, "extractoFinanceiro")))
                driver.execute_script("arguments[0].click();", link_det)
                
                tabela_apareceu = False
                for i in range(10): # Tenta 10 vezes (2 segundos cada)
                    texto_temp = obter_texto_total(driver)
                    if "Descrição" in texto_temp or "Crédito" in texto_temp or "Débito" in texto_temp:
                        tabela_apareceu = True
                        break
                    time.sleep(2)
                
                if tabela_apareceu:

                    detalhes = obter_texto_total(driver)

                    if len(detalhes) < 200:
                        linhas = driver.find_elements(By.TAG_NAME, "tr")
                        detalhes += "\n--- REFORÇO TR ---\n"
                        for tr in linhas:
                            detalhes += tr.text.replace("\n", " | ") + "\n"

                    texto_final += f"\n\n=== DETALHE EXTRACTO ===\n{detalhes}"

                driver.back()
                time.sleep(3)

            except Exception as e:
                print(f"Erro")

        # Configurar Alertas
        elif filename == "configurar_alertas":
            try:
                tab = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//span[text()='Consultas']")))
                driver.execute_script("arguments[0].click();", tab)
                time.sleep(5)
                texto_final += f"\n=== CONSULTAS ===\n{obter_texto_total(driver)}"
            except: pass

        # Inscrição Exames e Notas
        elif filename in ["inscricao_exames", "notas"]:
            texto_final += obter_texto_total(driver)
            if filename in DOWNLOAD_MAP:
                tratar_interacoes_e_downloads(driver, filename, DOWNLOAD_MAP[filename])
            
            nome_aba = "Definitivas" if filename == "inscricao_exames" else "ECTS"
            try:
                tab = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, f"//span[text()='{nome_aba}']")))
                driver.execute_script("arguments[0].click();", tab)
                time.sleep(5)
                texto_final += f"\n\n=== {nome_aba.upper()} ===\n{obter_texto_total(driver)}"
                
                botoes = driver.find_elements(By.XPATH, "//button[@data-qtip='Exportar para Excel']")
                if botoes:
                    btn_alvo = botoes[-1] if filename == "inscricao_exames" else [b for b in botoes if b.is_displayed()][0]
                    driver.execute_script("arguments[0].click();", btn_alvo)
                    time.sleep(3)
            except: pass

        # Others
        else:
            texto_final += obter_texto_total(driver)
            if filename in DOWNLOAD_MAP:
                tratar_interacoes_e_downloads(driver, filename, DOWNLOAD_MAP[filename])

        final_result.append({
            "nome": filename,
            "xpath": xpath,
            "conteudo": texto_final 
        })
        print(f"   📄 Guardado no JSON.")

    except Exception as e:
        print(f"Erro")

    for _ in range(back_times):
        if "Pedidos de Revisão" in xpath:
            driver.back()
            time.sleep(1)
        driver.back()
        time.sleep(2)
        fechar_popup_later(driver)

# Login
try:
    driver.get('https://netpa.novaims.unl.pt/netpa/page?stage=difhomestage')
    time.sleep(3)
    try: WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Aceitar todos')]"))).click()
    except: pass

    WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "loginregisterLink"))).click()
    WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, "//input[contains(@placeholder, 'utilizador')]"))).send_keys("******")  # The netpa username must be inserted here, but is hidden for security reasons. You can try it by looking to the secrets/.env file.
    pi = driver.find_element(By.XPATH, "//input[contains(@placeholder, 'palavra-chave')]")
    pi.send_keys("*****")  # The netpa password must be inserted here, but is hidden for security reasons. You can try it by looking to the secrets/.env file.
    driver.execute_script("arguments[0].click();", WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Entrar')]"))))
    
    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.CLASS_NAME, "menuItemLink")))
    fechar_popup_later(driver)
    print("Login OK.")

    links_e_nomes = [
        ("//a[contains(text(), 'Matrículas / Inscrições')]", "matricula_inscricao"),
        ("//a[contains(@href, 'ListarAulasAluno')]", "sumarios"),
        ("//a[contains(@href, 'ConsultaInscricaoEpocas') and text()='Inscrição Exames']", "inscricao_exames"),
        ("//a[contains(@href, 'ConsultaNotasAluno') and contains(text(), 'Consulta')]", "notas"),
        ("//a[contains(@href, 'SituacaoFinanceira') and contains(text(), 'Financeiro')]", "resumo_financeiro"),
        ("//a[contains(@href, 'BoletimMatricula') and contains(text(), 'Dados Pessoais')]", "dados_pessoais"),
        ("//a[contains(@href, 'myconsents')]", "consentimentos"),
        ("//a[contains(@href, 'DIFTasks') and contains(text(), 'Configurar Alertas')]", "configurar_alertas"),
        ("//a[contains(text(), 'Calendário de candidaturas')]", "calendario_candidaturas"),
        ("//a[contains(text(), 'Candidatura a um curso')]", "candidatura_curso"),
        ("//a[contains(text(), 'Consultar candidaturas')]", "consultar_candidaturas"),
        ("//a[contains(text(), 'Plano de Estudos')]", "plano_estudos"),
        ("//a[contains(text(), 'Situação Curricular')]", "situacao_curricular"),
        ("//a[contains(text(), 'Bolsa')]", "bolsa"),
        ("//a[contains(text(), 'Horário')]", "horario"),
        ("//a[contains(text(), 'Faltas')]", "faltas"),
        ("//a[contains(text(), 'Formação Avançada')]", "formacao_avancada"),
        ("//a[contains(text(), 'Estágios')]", "estagios"),
        ("//a[contains(text(), 'Pedidos de Revisão de Notas')]", "pedidos_revisao_notas"),
        ("//a[contains(text(), 'Requisições de Documentos')]", "requisicoes_documentos"),
        ("//a[contains(text(), 'Atividades curriculares')]", "atividades_curriculares"),
        ("//a[contains(text(), 'Atividades extracurriculares')]", "atividades_extracurriculares"),
        ("//a[contains(text(), 'Competências linguísticas')]", "competencias_linguisticas"),
        ("//a[contains(text(), 'Propinas')]", "propinas"),
        ("//a[contains(text(), 'Dívidas')]", "dividas"),
        ("//a[contains(text(), 'Referências ')]", "referencias_mb"),
        ("//a[contains(text(), 'Documentos faturação')]", "documentos_faturacao"),
        ("//a[contains(text(), 'Pagamentos')]", "pagamentos"),
        ("//a[contains(text(), 'Associação de Entidades Pagadoras')]", "associacao_entidades_pagadoras"),
        ("//a[contains(text(), 'Planos de Pagamento')]", "planos_pagamento")
    ]

    final_result = []
    for xpath, nome in links_e_nomes:
        click_and_dump(driver, xpath, nome)

    with open("netpa_dump_2.0.json", "w", encoding="utf-8") as f:
        json.dump(final_result, f, ensure_ascii=False, indent=2)

except Exception as e:
    print(f"Erro")
finally:
    driver.quit()

  A iniciar browser...
  Login...
Login OK.
  Entrou em 'matricula_inscricao'
   Processar: matricula_inscricao...
        Encontrados 3 botões/itens.
       URL PDF: https://netpa.novaims.unl.pt/n...
        Guardado: matricula_inscricao_1.pdf
       URL PDF: https://netpa.novaims.unl.pt/n...
        Guardado: matricula_inscricao_2.pdf
       URL PDF: https://netpa.novaims.unl.pt/n...
        Guardado: matricula_inscricao_3.pdf
     Guardado no JSON.
  Entrou em 'sumarios'
     Guardado no JSON.
  Entrou em 'inscricao_exames'
   Processar: inscricao_exames...
        Encontrados 1 botões/itens.
        Download 1 iniciado.
     Aba Definitivas...
     Guardado no JSON.
  Entrou em 'notas'
   Processar: notas...
        Encontrados 1 botões/itens.
        Download 1 iniciado.
     Aba ECTS...
     Guardado no JSON.
  Entrou em 'resumo_financeiro'
     Entrar em 'Ver detalhe extracto'...
     A aguardar tabela (procurando 'Descrição')...
        Tabela detetada na tentativa 1!
        D